# 01 — Matching baselines (exploration)

This notebook is for exploration and inspection only. It **reads saved results**; it never fits a model or touches evaluation points.
Regenerate everything with `python -m mapalign.run_all`. Reusable code lives in `src/mapalign/`.

In [ ]:
from mapalign.config import load_config, resolve_path
from mapalign.io import load_image, file_hash, read_json
import pandas as pd

cfg = load_config()
hist_path = resolve_path(cfg['historical_map']['path'])
assert file_hash(hist_path) == cfg['historical_map']['sha256'], 'input differs from the verified scan'
res = resolve_path(cfg['experiment']['output_dir'])

## Automatic matching: counts and diagnostics

In [ ]:
rows = []
for m in ('sift', 'loftr'):
    r = read_json(res / f'{m}_run.json')
    rows.append({'method': m, 'status': r['status'], 'matches': r['estimation']['input_correspondences'],
                 'inliers': r['estimation']['inliers'], 'inlier_ratio': r['estimation']['inlier_ratio'],
                 'corner_vs_graticule_m': (r['graticule_corner_check'] or {}).get('mean_distance_m'),
                 'tile_upper_bound': r['prior_diagnostics'].get('aligned_tile_upper_bound', {}).get('correct_fraction')})
pd.DataFrame(rows)

In [ ]:
from IPython.display import Image, display
for f in ('sift_matches.png', 'loftr_matches.png', 'overlay_loftr.png'):
    display(Image(filename=str(res / f), width=900))

## Held-out evaluation

In [ ]:
m = pd.read_csv(res / 'metrics.csv')
m[['model', 'method', 'method_status', 'n_valid', 'median_m', 'mean_m', 'rmse_m', 'max_m', 'p90_m']]

In [ ]:
err = pd.read_csv(res / 'evaluation_errors.csv')
err[err.model == 'homography'].pivot(index='point_id', columns='method', values='error_m')

In [ ]:
display(Image(filename=str(res / 'per_point_errors.png'), width=900))